# Phase 2 Pipeline

Classifier -> OCR -> BioBERT NER -> Gemini (NER-grounded prompts).

Runs on a caller-supplied list of test images and saves each result to a
JSON file for comparison against Phase 1 (`phase1_pipeline.ipynb`), which
skips NER and prompts Gemini with OCR text only.

In [ ]:
import json
import sys
from pathlib import Path

import cv2
import numpy as np
from PIL import Image, ImageOps

# Walk up from the kernel's cwd to find the project root, since VS Code's
# Jupyter kernel uses the notebook's own folder as cwd (not the project
# root) when the notebook lives in a subfolder like pipelines/.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "config.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate project root (config.py not found in any parent directory)")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import TEST_IMAGES_PATH
from models import DocumentClassifier, OCREngine, NERExtractor
from models.image_preprocessor import ALLOWED_EXTENSIONS, MAX_FILE_SIZE_BYTES, BLUR_VARIANCE_THRESHOLD
from services.gemini_summarizer_test import GeminiSummarizer

print("Loading OCREngine (PaddleOCR) ...")
ocr_engine = OCREngine()

print("Loading DocumentClassifier (LayoutLMv3) ...")
classifier = DocumentClassifier(ocr_engine=ocr_engine)

print("Loading NERExtractor (BioBERT) ...")
ner_extractor = NERExtractor()

print("Loading GeminiSummarizer ...")
summarizer = GeminiSummarizer()

print("All models loaded.")

c:\Users\PRIYANKA\Desktop\MedAi\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading OCREngine (PaddleOCR) ...


c:\Users\PRIYANKA\Desktop\MedAi\.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Loading DocumentClassifier (LayoutLMv3) ...


Loading weights: 100%|██████████| 216/216 [00:00<00:00, 4514.05it/s]


Loading NERExtractor (BioBERT) ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4425.40it/s]


Loading GeminiSummarizer ...
All models loaded.


In [2]:
def check_image_quality(image_path):
    """Validate that an image can be opened and score its sharpness.

    Same as Phase 1: runs the same checks as
    models/image_preprocessor.load_image_for_ocr (file exists, extension
    allowed, size limit, not corrupted) but never raises on blur -- a
    blurry image is flagged via low_quality=True so the pipeline still
    attempts classification/OCR on it instead of rejecting it outright. If
    classification/OCR later fails anyway (e.g. the image is too blurry
    for PaddleOCR's own stricter check), that is caught as a normal
    pipeline failure further down.

    Returns a dict: valid, low_quality, blur_variance, error
    """
    path = Path(image_path)

    if not path.exists():
        return {"valid": False, "low_quality": False, "blur_variance": None,
                "error": f"Image file not found: {image_path}"}

    if path.suffix.lower() not in ALLOWED_EXTENSIONS:
        return {"valid": False, "low_quality": False, "blur_variance": None,
                "error": f"Unsupported file type '{path.suffix}'. Allowed: {sorted(ALLOWED_EXTENSIONS)}"}

    file_size = path.stat().st_size
    if file_size > MAX_FILE_SIZE_BYTES:
        return {"valid": False, "low_quality": False, "blur_variance": None,
                "error": f"Image file too large ({file_size / 1_000_000:.1f} MB). "
                         f"Max allowed is {MAX_FILE_SIZE_BYTES / 1_000_000:.0f} MB."}

    try:
        image = Image.open(path)
        image = ImageOps.exif_transpose(image).convert("RGB")
    except Exception as e:
        return {"valid": False, "low_quality": False, "blur_variance": None,
                "error": f"Could not read image file: {e}"}

    gray = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2GRAY)
    blur_variance = round(float(cv2.Laplacian(gray, cv2.CV_64F).var()), 2)
    low_quality = blur_variance < BLUR_VARIANCE_THRESHOLD

    return {"valid": True, "low_quality": low_quality, "blur_variance": blur_variance, "error": None}

In [3]:
def call_gemini(doc_type, raw_text, entities_by_type):
    """Dispatch to the prescription or report Gemini prompt, grounded with
    NER entities (drugs/dosage/frequency/duration for prescriptions; test
    names/values/ranges for reports), based on doc_type.

    Returns a dict: summary, doc_type, error (error is None on success).
    """
    if doc_type == "prescription":
        return summarizer.summarize_with_entities("prescription", raw_text, entities_by_type)
    elif doc_type == "report":
        return summarizer.summarize_with_entities("report", raw_text, entities_by_type)
    else:
        return {"summary": None, "doc_type": doc_type,
                "error": f"No Gemini prompt for doc_type '{doc_type}'"}

In [4]:
def run_phase2_pipeline(image_path):
    """Run Phase 2 (Classifier -> OCR -> NER -> Gemini) on a single image."""
    filename = Path(image_path).name

    quality = check_image_quality(image_path)
    if not quality["valid"]:
        return {"image": filename, "status": "failed", "message": quality["error"]}

    try:
        classify_result = classifier.predict(image_path)
    except Exception as e:
        return {
            "image": filename,
            "status": "failed",
            "message": f"Classification/OCR failed: {e}",
            "low_quality": quality["low_quality"],
        }

    doc_type = classify_result["doc_type"]
    classifier_confidence = classify_result["confidence"]

    if doc_type == "non_medical":
        return {
            "image": filename,
            "status": "rejected",
            "message": "Please upload a valid medical document",
            "doc_type": "non_medical",
            "classifier_confidence": classifier_confidence,
            "low_quality": quality["low_quality"],
        }

    # classifier.predict() already ran OCR internally (DocumentClassifier
    # shares the OCREngine instance) -- reuse its raw_text instead of
    # triggering a second OCR pass over the same image
    raw_text = classify_result["raw_text"]
    ocr_confidence = classify_result["ocr_avg_confidence"]

    if not raw_text.strip():
        return {
            "image": filename,
            "status": "failed",
            "message": "No text extracted",
            "doc_type": doc_type,
            "classifier_confidence": classifier_confidence,
            "ocr_confidence": ocr_confidence,
            "low_quality": quality["low_quality"],
        }

    # NERExtractor returns entity_count=0 / entities=[] for text with no
    # matches rather than raising, so empty entities flow through to Gemini
    # unchanged -- no special-casing needed here
    ner_result = ner_extractor.extract_entities(raw_text)
    entities = ner_result["entities"]
    entity_count = ner_result["entity_count"]
    ner_avg_confidence = ner_result["avg_confidence"]
    entities_by_type = ner_extractor.get_entities_by_type(ner_result)

    gemini_result = call_gemini(doc_type, raw_text, entities_by_type)
    if gemini_result["error"]:
        return {
            "image": filename,
            "status": "failed",
            "message": gemini_result["error"],
            "doc_type": doc_type,
            "classifier_confidence": classifier_confidence,
            "ocr_confidence": ocr_confidence,
            "ner_avg_confidence": ner_avg_confidence,
            "entity_count": entity_count,
            "low_quality": quality["low_quality"],
        }

    return {
        "image": filename,
        "status": "success",
        "doc_type": doc_type,
        "classifier_confidence": classifier_confidence,
        "ocr_confidence": ocr_confidence,
        "ner_avg_confidence": ner_avg_confidence,
        "entity_count": entity_count,
        "raw_text": raw_text,
        "entities": entities,
        "summary": gemini_result["summary"],
        "low_quality": quality["low_quality"],
    }

In [5]:
def run_batch(image_list, output_filename):
    """Run Phase 2 on a list of image paths.

    Prints one progress line per image and appends each result to
    output_filename (a JSON list), saving after every image so a crash or
    interrupt mid-batch doesn't lose already-processed results. Creates the
    file if it doesn't exist yet.
    """
    output_path = Path(output_filename)
    if output_path.exists():
        with open(output_path, encoding="utf-8") as f:
            results = json.load(f)
    else:
        results = []

    for image_path in image_list:
        filename = Path(image_path).name
        try:
            result = run_phase2_pipeline(image_path)
        except Exception as e:
            result = {"image": filename, "status": "failed", "message": str(e)}

        doc_type_str = str(result.get("doc_type", "-"))
        entity_count_str = str(result.get("entity_count", "-"))
        print(f"{filename:20s} doc_type={doc_type_str:15s} entity_count={entity_count_str:5s} status={result['status']}")
        results.append(result)

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"\nDone. {len(results)} total results saved to {output_path}")
    return results

In [ ]:
# Pass whichever images to run each time, e.g.:
my_image_list = [
    str(Path(TEST_IMAGES_PATH) / "001img.jpg"),
    str(Path(TEST_IMAGES_PATH) / "003img.jpg"),
    str(Path(TEST_IMAGES_PATH) / "006img.png"),
]

results = run_batch(my_image_list, "phase2_results.json") 

In [ ]:
# Pass whichever images to run each time, e.g.:
my_image_list = [
    str(Path(TEST_IMAGES_PATH) / "017img.png"),
    str(Path(TEST_IMAGES_PATH) / "018img.jpg"),
    str(Path(TEST_IMAGES_PATH) / "019img.jpeg"),
    str(Path(TEST_IMAGES_PATH) / "020img.jpg"),
    str(Path(TEST_IMAGES_PATH) / "021img.png"),
    str(Path(TEST_IMAGES_PATH) / "022img.jpeg"),
    str(Path(TEST_IMAGES_PATH) / "021img.png"),
    str(Path(TEST_IMAGES_PATH) / "022img.jpeg"),
    str(Path(TEST_IMAGES_PATH) / "023img.jpg"),
    str(Path(TEST_IMAGES_PATH) / "024img.webp"),
    str(Path(TEST_IMAGES_PATH) / "025img.jpeg"),
    str(Path(TEST_IMAGES_PATH) / "026img.jpeg"),
    str(Path(TEST_IMAGES_PATH) / "027img.png")
]

results = run_batch(my_image_list, "phase2_results.json")

007img.jpg           doc_type=non_medical     entity_count=-     status=rejected
008img.jpeg          doc_type=prescription    entity_count=20    status=success
009img.jpg           doc_type=prescription    entity_count=35    status=success
010img.jpg           doc_type=report          entity_count=34    status=success
011img.png           doc_type=report          entity_count=9     status=success
012img.jpg           doc_type=report          entity_count=10    status=success
013img.jpg           doc_type=prescription    entity_count=48    status=success
014img.jpeg          doc_type=prescription    entity_count=28    status=success
015img.jpeg          doc_type=report          entity_count=18    status=success
016img.jpg           doc_type=report          entity_count=21    status=success

Done. 16 total results saved to phase2_results.json
